In [ ]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import utils
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import pandas as pd
import sagemaker, boto3, os
import paths, utils
from utils import Sql, train_val_test_split, get_sm_service_role_arn

import sagemaker.core.helper.session_helper as sm_session_helper
import sagemaker.core.image_uris as sm_image_uris
import sagemaker.core.inputs as sm_inputs
import sagemaker.core.resources as sm_resources
import sagemaker.core.shapes as sm_shapes
from   sagemaker.core.training import configs as sm_train_configs
import sagemaker.core.transformer as sm_transformer

import sagemaker.train.model_trainer as sm_model_trainer






In [2]:
# User vars
region = 'us-east-1'
model_package_group_name='abalone'
model_version=1
target_name='rings'
prediction_name=target_name+'_prediction'

p=paths.Paths(bucket_name='omm-test-bucket', project_name='abalone', model_prefix='abalone')
role = get_sm_service_role_arn()
boto_session=boto3.Session(region_name=region)
sagemaker_session = sm_session_helper.Session(boto_session=boto_session)

[06/14/26 23:15:16] INFO     Found credentials from IAM Role: ec2-1                             ]8;id=9927112;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=9927113;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\

In [3]:
# DATA INGESTION
sql=Sql('user-1', 'password', 'db_1')
abalone_df = sql.query('SELECT * FROM abalone;')
abalone_df=abalone_df.drop(columns=['id','created_at'])

In [4]:
# FEATURE ENGINEERING
abalone_df = pd.get_dummies(abalone_df, columns=['sex'], drop_first=False)
abalone_df[['sex_I', 'sex_M', 'sex_F']] = abalone_df[['sex_I', 'sex_M', 'sex_F']].astype(int)
abalone_df.head()

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings,sex_F,sex_I,sex_M
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,0,0,1
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,0,0,1
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,1,0,0
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,0,0,1
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,0,1,0


In [ ]:
# DATA FORMATTING
y = abalone_df['rings']
X = abalone_df.drop(columns=['rings'])

X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y, train_size=0.7, val_size=0.15, random_state=42)

pd.concat([y_train, X_train], axis=1).to_csv(p.train_file, index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv(p.validation_file, index=False, header=False)
pd.concat([y_test, X_test], axis=1).to_csv(p.test_file, index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv(p.baseline_file, index=False, header=True) # need header

# For development and testing
X_test.to_csv(p.test_X_file, index=True, header=False)
y_test.to_csv(p.test_y_file, index=True, header=False)

input_data_config = [
    sm_train_configs.InputData(channel_name='train', data_source=p.train_dir, content_type='text/csv'),
    sm_train_configs.InputData(channel_name='validation', data_source=p.validation_dir, content_type='text/csv')
]

In [9]:
model_trainer = sm_model_trainer.ModelTrainer(
    training_image=sm_image_uris.retrieve('xgboost', region, version='1.5-1'),
    hyperparameters={
        'objective': 'reg:squarederror',
        'num_round': '100',
        'max_depth': '5',
        'eta': '0.2',
        'subsample': '0.8',
        'eval_metric': 'rmse'
    },
    role=role,
    sagemaker_session=sagemaker_session,
    base_job_name='abalone-train-job',
    output_data_config=sm_shapes.OutputDataConfig(s3_output_path=p.model_dir),
    networking=sm_train_configs.Networking(
        subnets=['subnet-001be661bcef4b615', 'subnet-003ad32933ca43e74'],
        security_group_ids=['sg-00f14515abe1e47e8']
    )
)

[06/14/26 23:20:56] INFO     Ignoring unnecessary instance type: None.                            ]8;id=9927308;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=9927309;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    INFO     Compute not provided. Using default:                                   ]8;id=9927314;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=9927315;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py#102\102]8;;\
                             instance_type='ml.m5.xlarge' instance_count=1 volume_size_in_gb=30                    
                             volume_kms_key_id=None keep_alive_period_in_seconds=None                              
                             instance_groups=None training_plan_arn=None                                           
                             instance_placement_config=None enable_managed_spot_training=None                      

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=9927320;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=9927321;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     Training image URI:                                               ]8;id=9927326;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=9927327;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.                     
                             5-1                                                                                   

In [14]:
dir(model_trainer)

['CONFIGURABLE_ATTRIBUTES',
 'SERIALIZABLE_CONFIG_ATTRIBUTES',
 '__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__del__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '

In [10]:
training_job = model_trainer.train(
    input_data_config=input_data_config,
    wait=True,
    logs=True
)

[06/14/26 23:20:59] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=9927332;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=9927333;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Creating training_job resource.                                     ]8;id=9927338;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9927339;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31116\31116]8;;\

/home/ec2-user/.local/lib/python3.9/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

[06/14/26 23:23:21] INFO     abalone-train-job-20260614232059/algo-1-1781479308:                 ]8;id=9927344;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9927345;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             /miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36:                         
                             FutureWarning: pandas.Int64Index is deprecated and will be removed                    
                             from pandas in a future version. Use pandas.Index with the                            
                             appropriate dtype instead.                                                            
                               from pandas import MultiIndex, Int64Index                                           

                    INFO     abalone-train-job-20260614232059/algo-1-1781479308:                 ]8;id=9927350;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9927351;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [2026-06-14 23:23:08.407 ip-172-31-107-250.ec2.internal:7 INFO                        
                             utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None                                      

...

                    INFO     abalone-train-job-20260614232059/algo-1-1781479308:                 ]8;id=9928040;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928041;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [96]#011train-rmse:1.25616#011validation-rmse:2.02881                                 

                    INFO     abalone-train-job-20260614232059/algo-1-1781479308:                 ]8;id=9928046;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928047;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [97]#011train-rmse:1.25277#011validation-rmse:2.02753                                 

                    INFO     abalone-train-job-20260614232059/algo-1-1781479308:                 ]8;id=9928052;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928053;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [98]#011train-rmse:1.24839#011validation-rmse:2.02505                                 

                    INFO     abalone-train-job-20260614232059/algo-1-1781479308:                 ]8;id=9928058;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928059;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [99]#011train-rmse:1.23891#011validation-rmse:2.02563                                 

[06/14/26 23:23:32] INFO     Final Resource Status: Completed                                    ]8;id=9928064;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928065;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31442\31442]8;;\

In [11]:
# BASELINING
# get baseline X
baseline=pd.read_csv(p.baseline_file, header=0)
baseline[target_name] = baseline[target_name].astype(float)
baseline_X = baseline.drop(columns=[target_name])
baseline_X.to_csv(p.baseline_X_file, index=False, header=False)



In [ ]:


model_name = 'abalone-1'

model = sm_resources.Model.create(
    model_name=model_name,
    execution_role_arn=role,  # plain string ARN, not ParameterString
    containers=[
        sm_shapes.shapes.ContainerDefinition(
            image=sm_image_uris.retrieve('xgboost', region, version='1.5-1'),
            model_data_url='s3://omm-test-bucket/abalone/models/abalone/model/abalone-train-job-20260614232059/output/model.tar.gz' # parameterize this. for some reason training_job returns none
        )
    ],
    session=boto_session
)

[06/14/26 23:43:57] INFO     Ignoring unnecessary instance type: None.                            ]8;id=9928088;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=9928089;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    INFO     Creating model resource.                                            ]8;id=9928095;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928096;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#20477\20477]8;;\

In [20]:
transformer=sm_transformer.Transformer(
    model_name=model.model_name, 
    instance_count=1, 
    instance_type='ml.m5.xlarge', 
    output_path=p.temp_data_dir,
    base_transform_job_name='abalone-transform-job',
    sagemaker_session=sagemaker_session
    )

In [24]:
# Get predictions from baseline X
transformer.transform(data=p.baseline_X_file, content_type='text/csv', split_type='Line')
transformer.wait()



[06/14/26 23:57:46] INFO     Creating transform job with name:                                   ]8;id=9928122;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/transformer.py\transformer.py]8;;\:]8;id=9928123;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/transformer.py#403\403]8;;\
                             abalone-transform-job-2026-06-14-23-57-46-759                                         

/home/ec2-user/.local/lib/python3.9/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

[06/15/26 00:02:35] INFO     abalone-transform-job-2026-06-14-23-57-46-759/i-0cd6a5c8a5e6f76bd-1 ]8;id=9928129;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928130;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32324\32324]8;;\
                             781481661:                                                                            
                             /miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36:                         
                             FutureWarning: pandas.Int64Index is deprecated and will be removed                    
                             from pandas in a future version. Use pandas.Index with the                            
                             appropriate dtype instead.                                                            
                               from pandas import MultiIndex, Int64Index                                           

                    INFO     abalone-transform-job-2026-06-14-23-57-46-759/i-0cd6a5c8a5e6f76bd-1 ]8;id=9928135;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928136;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32324\32324]8;;\
                             781481661:                                                                            
                             [2026-06-15:00:02:25:INFO] No GPUs detected (normal if no gpus                        
                             installed)                                                                            

                    INFO     abalone-transform-job-2026-06-14-23-57-46-759/i-0cd6a5c8a5e6f76bd-1 ]8;id=9928141;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928142;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32324\32324]8;;\
                             781481661:                                                                            
                             [2026-06-15:00:02:25:INFO] No GPUs detected (normal if no gpus                        
                             installed)                                                                            

...

                    INFO     abalone-transform-job-2026-06-14-23-57-46-759/i-0cd6a5c8a5e6f76bd-1 ]8;id=9928627;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928628;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32324\32324]8;;\
                             781481661:                                                                            
                             169.254.255.130 - - [15/Jun/2026:00:02:31 +0000] "GET                                 
                             /execution-parameters HTTP/1.1" 200 84 "-" "Go-http-client/1.1"                       

                    INFO     abalone-transform-job-2026-06-14-23-57-46-759/i-0cd6a5c8a5e6f76bd-1 ]8;id=9928633;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928634;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32324\32324]8;;\
                             781481661:                                                                            
                             [2026-06-15:00:02:31:INFO] Determined delimiter of CSV input is ','                   

                    INFO     abalone-transform-job-2026-06-14-23-57-46-759/i-0cd6a5c8a5e6f76bd-1 ]8;id=9928639;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928640;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32324\32324]8;;\
                             781481661:                                                                            
                             /miniconda3/lib/python3.8/site-packages/xgboost/core.py:105:                          
                             UserWarning: ntree_limit is deprecated, use `iteration_range` or                      
                             model slicing instead.                                                                
                               warnings.warn(                                                                      

                    INFO     abalone-transform-job-2026-06-14-23-57-46-759/i-0cd6a5c8a5e6f76bd-1 ]8;id=9928645;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928646;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32324\32324]8;;\
                             781481661:                                                                            
                             169.254.255.130 - - [15/Jun/2026:00:02:31 +0000] "POST /invocations                   
                             HTTP/1.1" 200 13629 "-" "Go-http-client/1.1"                                          

                    INFO     Final Resource Status: Completed                                    ]8;id=9928651;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9928652;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#32327\32327]8;;\

In [25]:
# Rename/move predictions file
# utils.move_s3_file(boto_session, p.baseline_model_out_file, p.baseline_y_file)
utils.move_s3_file(boto_session, p.baseline_model_out_file, f'{p.baseline_dir}/baseline_y.csv')

In [ ]:
# Register model from estimator
model = estimator.create_model(name=f"{model_package_group_name}-{str(model_version)}")

model.create(instance_type='ml.m5.large') # temporary instance for validation

model_package = model.register(
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.large'],
    transform_instances=['ml.m5.large'],
    model_package_group_name='abalone',
    description='XGBoost model for abalone ring prediction',
    approval_status='PendingManualApproval'  # or 'Approved'
)

INFO:sagemaker:Creating model with name: abalone-1
